Retrieval Augmented Generation (RAG) Bot

In [1]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

"""""
img = mpimg.imread('process.png')
plt.imshow(img)
plt.axis('off')
"""

'""\nimg = mpimg.imread(\'process.png\')\nplt.imshow(img)\nplt.axis(\'off\')\n'

Content Extraction 

In [2]:
import pdfplumber
import os
from huggingface_hub import login
from openai import OpenAI
from langchain.vectorstores import FAISS
import fitz  # PyMuPDF for PDFs
import os

PDF_FOLDER = os.path.join(os.getcwd(), 'raw_documents')
OUTPUT_FOLDER = os.path.join(os.getcwd(), "extracted_texts")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

def extract_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text += (page.extract_text() or "") + "\n"
    return text

for filename in os.listdir(PDF_FOLDER):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(PDF_FOLDER, filename)
        print("Extracting:", pdf_path)

        text = extract_pdf(pdf_path)

        # Save text file for this PDF
        output_path = os.path.join(OUTPUT_FOLDER, filename.replace(".pdf", ".txt"))
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(text)



c:\Users\Migs\Desktop\ragbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Extracting: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\raw_documents\Topic 1_ Pre-16th Century Philippines Reading Materials.pdf


Chunking

In [5]:
import os
import json
from langchain_experimental.text_splitter import SemanticChunker
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import TokenTextSplitter

# ============================================================
#                   HYPERPARAMETER CONFIG
# ============================================================
CONFIG = {
    "model_name": "sentence-transformers/all-MiniLM-L6-v2",

    # Semantic Chunking Parameters
    "semantic_min_chunk_size":        200,
    "semantic_breakpoint_type":       "percentile",
    "semantic_breakpoint_amount":     95,

    # Token-based Chunking Parameters
    "token_chunk_size":   250,
    "token_overlap":       50,
    "tokenizer_name":     "cl100k_base",

    # Paths
    "text_folder":   os.path.join(os.getcwd(), "extracted_texts"),
    "chunks_folder": os.path.join(os.getcwd(), "chunks_json"),
}
os.makedirs(CONFIG["chunks_folder"], exist_ok=True)
# ============================================================


# -------------------- LOAD EMBEDDINGS --------------------
embeddings = HuggingFaceEmbeddings(
    model_name=CONFIG["model_name"]
)

# -------------------- SEMANTIC CHUNKER --------------------
semantic_chunker = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type=CONFIG["semantic_breakpoint_type"],
    breakpoint_threshold_amount=CONFIG["semantic_breakpoint_amount"],
    min_chunk_size=CONFIG["semantic_min_chunk_size"],
)

# -------------------- TOKEN SPLITTER --------------------
token_splitter = TokenTextSplitter(
    chunk_size=CONFIG["token_chunk_size"],
    chunk_overlap=CONFIG["token_overlap"],
    encoding_name=CONFIG["tokenizer_name"]
)

# ============================================================
#                   PROCESSING LOOP
# ============================================================
for filename in os.listdir(CONFIG["text_folder"]):
    if filename.endswith(".txt"):
        file_path = os.path.join(CONFIG["text_folder"], filename)
        print("Chunking:", file_path)

        # Load extracted text
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        # Stage 1: Semantic chunking
        semantic_chunks = semantic_chunker.split_text(text)

        # Stage 2: Token chunking
        final_chunks = []
        for sc in semantic_chunks:
            final_chunks.extend(token_splitter.split_text(sc))

        # Save JSON output
        base_name = filename.replace(".txt", "")
        json_path = os.path.join(CONFIG["chunks_folder"], f"{base_name}_chunks.json")

        json_output = [
            {
                "doc_id": base_name,
                "chunk_id": f"{base_name}_chunk_{i}",
                "chunk_index": i,
                "text": chunk
            }
            for i, chunk in enumerate(final_chunks)
        ]

        with open(json_path, "w", encoding="utf-8") as out:
            json.dump(json_output, out, indent=4, ensure_ascii=False)

        print(f"Saved {len(final_chunks)} chunks → {json_path}")


Chunking: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\extracted_texts\Topic 1_ Pre-16th Century Philippines Reading Materials.txt
Saved 25 chunks → c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\chunks_json\Topic 1_ Pre-16th Century Philippines Reading Materials_chunks.json


In [6]:
import os
import json
import re
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

from together import Together  # <-- Together.ai client

# -------------------- CONFIG --------------------
CHUNKS_FOLDER = "chunks_json"
EVAL_OUTPUT = "together_rag_eval_dataset.json"
TOP_K = 5  # number of similar chunks to consider
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TOGETHER_API_KEY = "f093074f102974466d625db36d8bd171b92df916fa78eb7b91faa9108e6ed5c2"
TOGETHER_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct-Lite"  # cheap hosted model
MAX_OUTPUT_TOKENS = 150  # limit token output per chunk

os.makedirs(CHUNKS_FOLDER, exist_ok=True)

# -------------------- LOAD CHUNKS --------------------
all_chunks_text = []
all_chunks_meta = []

for file_name in os.listdir(CHUNKS_FOLDER):
    if file_name.endswith(".json"):
        path = os.path.join(CHUNKS_FOLDER, file_name)
        with open(path, "r", encoding="utf-8") as f:
            chunks = json.load(f)
        for chunk in chunks:
            all_chunks_text.append(chunk["text"])
            all_chunks_meta.append({
                "doc_id": chunk.get("doc_id"),
                "chunk_id": chunk.get("chunk_id"),
                "chunk_index": chunk.get("chunk_index")
            })

print(f"Loaded {len(all_chunks_text)} chunks")

# -------------------- CREATE EMBEDDINGS --------------------
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
vectorstore = FAISS.from_texts(all_chunks_text, embeddings, metadatas=all_chunks_meta)

# -------------------- INITIALIZE TOGETHER CLIENT --------------------
client = Together(api_key=TOGETHER_API_KEY)

# -------------------- HELPER FUNCTION --------------------
def extract_json(text):
    """Extract JSON output from LLM text"""
    try:
        match = re.search(r'\[.*\]', text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except Exception as e:
        print("❌ JSON parsing error:", e)
    return []

# -------------------- BUILD EVAL DATASET --------------------
eval_dataset = []

for i, chunk_meta in enumerate(all_chunks_meta):
    query_text = all_chunks_text[i]

    # Step 1: Retrieve top-k similar chunks
    results = vectorstore.similarity_search(query_text, k=TOP_K)
    retrieved_chunks = [
        {"chunk_id": r.metadata["chunk_id"], "text": r.page_content}
        for r in results
    ]

    # Step 2: Prepare concatenated text for prompt
    chunks_text_for_prompt = "\n\n".join([f"Chunk ID: {c['chunk_id']}\nText: {c['text']}" 
                                          for c in retrieved_chunks])

    prompt = f"""
You are building a RAG evaluation dataset.

Here are some text chunks:

{chunks_text_for_prompt}

Task:
1. Generate 1 question answerable from these chunks.
2. Provide the answer.
3. Specify all chunk IDs necessary to answer the question.

Return output as JSON:
[{{"query": "...", "answer": "...", "relevant_chunks": ["..."]}}]
"""

    try:
        # Step 3: Call Together.ai hosted model
        response = client.chat.completions.create(
            model=TOGETHER_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_output_tokens=MAX_OUTPUT_TOKENS
        )

        text_output = response.choices[0].message.content
        qa_data = extract_json(text_output)
        if qa_data:
            eval_dataset.extend(qa_data)
            print(f"✅ Generated QA from chunk {chunk_meta['chunk_id']}")
        else:
            print(f"⚠️ No QA returned for chunk {chunk_meta['chunk_id']}")

    except Exception as e:
        print(f"❌ Error for chunk {chunk_meta['chunk_id']}: {e}")

# -------------------- SAVE DATASET --------------------
with open(EVAL_OUTPUT, "w", encoding="utf-8") as out:
    json.dump(eval_dataset, out, indent=4, ensure_ascii=False)

print(f"\n✅ Together.ai RAG evaluation dataset saved to {EVAL_OUTPUT}")
print(f"Total QA samples: {len(eval_dataset)}")



Loaded 2 chunks


C:\Users\Migs\AppData\Local\Temp\ipykernel_7976\3479689813.py:40: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
c:\Users\Migs\Desktop\ragbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Generated QA from chunk Topic 1_ Pre-16th Century Philippines Reading Materials_chunk_0
✅ Generated QA from chunk Topic 1_ Pre-16th Century Philippines Reading Materials_chunk_1

✅ Together.ai RAG evaluation dataset saved to together_rag_eval_dataset.json
Total QA samples: 2


Local Eval Dataset Creation

In [4]:
from langchain.llms import Ollama

llm = Ollama(model="mistral:instruct")

import json
import os
import re

CHUNKS_FOLDER = "chunks_json"
OUTPUT_FILE = "mistral_rag_eval.json"

def extract_json(text):
    try:
        match = re.search(r'\[.*\]', text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except Exception as e:
        print("❌ JSON parsing error:", e)
    return []

all_samples = []

for file_name in os.listdir(CHUNKS_FOLDER):
    if file_name.endswith(".json"):
        path = os.path.join(CHUNKS_FOLDER, file_name)
        with open(path, "r", encoding="utf-8") as f:
            chunks = json.load(f)
        
        for chunk in chunks:
            chunk_text = f"Chunk ID: {chunk['chunk_id']}\nText:\n{chunk['text']}\n"
            prompt = f"""
You are building a RAG evaluation dataset.

Here is one text chunk:

{chunk_text}

Task:
1. Generate 1 question answerable from this chunk.
2. Provide the answer.
3. Specify the chunk ID(s) necessary to answer.

Return output as JSON:
[{{"query": "...", "answer": "...", "relevant_chunks": ["{chunk['chunk_id']}"]}}]
"""
            try:
                response = llm(prompt)
                data = extract_json(response)
                if data:
                    all_samples.extend(data)
            except Exception as e:
                print(f"❌ Error for chunk {chunk['chunk_id']}: {e}")

with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
    json.dump(all_samples, out, indent=4, ensure_ascii=False)

print(f"\n✅ Mistral RAG evaluation dataset saved to {OUTPUT_FILE}")
print(f"Total samples: {len(all_samples)}")




C:\Users\Migs\AppData\Local\Temp\ipykernel_26340\1181058979.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="mistral:instruct")
C:\Users\Migs\AppData\Local\Temp\ipykernel_26340\1181058979.py:47: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = llm(prompt)


KeyboardInterrupt: 

Hosted Inference Eval Dataset Creation

In [5]:
from together import Together
import json
import os
import re

# -----------------------------
# Configuration
# -----------------------------
API_KEY = "f093074f102974466d625db36d8bd171b92df916fa78eb7b91faa9108e6ed5c2"
MODEL = "meta-llama/Meta-Llama-3-8B-Instruct-Lite"  # cheap, hosted inference
CHUNKS_FOLDER = "chunks_json"
OUTPUT_FILE = "together_rag_eval.json"
MAX_OUTPUT_TOKENS = 150  # limits response length per chunk

# Initialize Together client
client = Together(api_key=API_KEY)

# Function to extract JSON from text
def extract_json(text):
    try:
        match = re.search(r'\[.*\]', text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except Exception as e:
        print("❌ JSON parsing error:", e)
    return []

all_samples = []

# Iterate over chunk files
for file_name in os.listdir(CHUNKS_FOLDER):
    if file_name.endswith(".json"):
        path = os.path.join(CHUNKS_FOLDER, file_name)
        with open(path, "r", encoding="utf-8") as f:
            chunks = json.load(f)

        for chunk in chunks:
            chunk_text = f"Chunk ID: {chunk['chunk_id']}\nText:\n{chunk['text']}\n"
            prompt = f"""
You are building a RAG evaluation dataset.

Here is one text chunk:

{chunk_text}

Task:
1. Generate 1 question answerable from this chunk.
2. Provide the answer.
3. Specify the chunk ID(s) necessary to answer.

Return output as JSON:
[{{"query": "...", "answer": "...", "relevant_chunks": ["{chunk['chunk_id']}"]}}]
"""

            try:
                # Together.ai chat completion
                response = client.chat.completions.create(
                    model=MODEL,
                    messages=[{"role": "user", "content": prompt}],
                    max_output_tokens=MAX_OUTPUT_TOKENS
                )

                # Together returns choices[0].message.content
                text_output = response.choices[0].message.content
                data = extract_json(text_output)
                if data:
                    all_samples.extend(data)

            except Exception as e:
                print(f"❌ Error for chunk {chunk['chunk_id']}: {e}")

# Save all generated QA samples
with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
    json.dump(all_samples, out, indent=4, ensure_ascii=False)

print(f"\n✅ Together RAG evaluation dataset saved to {OUTPUT_FILE}")
print(f"Total samples: {len(all_samples)}")



✅ Together RAG evaluation dataset saved to together_rag_eval.json
Total samples: 2


Embedding

In [7]:
import os
import json
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document

# --------- PATHS ----------
CHUNKS_FOLDER = "chunks_json"
FAISS_FOLDER = "faiss_index"   # This will be a folder after saving

# --------- EMBEDDINGS ----------
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

# --------- LOAD CHUNK DATA ----------
documents = []

for filename in os.listdir(CHUNKS_FOLDER):
    if filename.endswith("_chunks.json"):
        file_path = os.path.join(CHUNKS_FOLDER, filename)
        print("Loading:", file_path)

        with open(file_path, "r", encoding="utf-8") as f:
            chunks = json.load(f)

        for chunk in chunks:
            doc = Document(
                page_content=chunk["text"],
                metadata={
                    "doc_id": chunk["doc_id"],
                    "chunk_id": chunk["chunk_id"],
                    "chunk_index": chunk["chunk_index"]
                }
            )
            documents.append(doc)

print(f"✅ Loaded {len(documents)} chunks as Documents")



Embedding model loaded.
Loading: chunks_json\Topic 1_ Pre-16th Century Philippines Reading Materials_chunks.json
✅ Loaded 25 chunks as Documents


Vector Store

In [8]:

from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings
)

vectorstore.save_local(FAISS_FOLDER)
print(f"✅ FAISS index saved to {FAISS_FOLDER}")


✅ FAISS index saved to faiss_index


Retrieval

In [10]:
import json
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

# ---- Load Embeddings (MUST MATCH INDEX) ----
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# ---- Load FAISS Vector Store ----
FAISS_FOLDER = "faiss_index"

vectorstore = FAISS.load_local(
    FAISS_FOLDER,
    embeddings,
    allow_dangerous_deserialization=True
)

# ---- Create Retriever ----
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 12
    }
)

# ---- Load Evaluation Queries ----
with open("rag_eval_dataset.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

# ---- Run Retrieval for Each Query ----
retrieval_results = []

for item in eval_data:
    query = item["query"]

    docs = retriever.get_relevant_documents(query)

    retrieval_results.append({
        "id": item.get("id"),
        "query": query,
        "retrieved_chunks": [
            {
                "content": doc.page_content,
                "metadata": doc.metadata
            }
            for doc in docs
        ]
    })

print(f"✅ Retrieved documents for {len(retrieval_results)} queries.")


# ---- Save Retrieval Output ----
OUTPUT_FILE = "retrieval_output.json"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(retrieval_results, f, indent=2, ensure_ascii=False)

print(f"✅ Retrieval results saved to {OUTPUT_FILE}")


C:\Users\Migs\AppData\Local\Temp\ipykernel_26340\858902306.py:38: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(query)


✅ Retrieved documents for 24 queries.
✅ Retrieval results saved to retrieval_output.json


Retrieval Evaluation

In [12]:
import json

# ----------------------------
# NORMALIZATION HELPER
# ----------------------------
def normalize(text: str) -> str:
    """Normalize text for safe lookup across JSON files."""
    if not isinstance(text, str):
        return ""
    return (
        text.strip()
            .lower()
            .replace("’", "'")
            .replace("“", '"')
            .replace("”", '"')
    )

# ----------------------------
# COMPUTE RECALL@K
# ----------------------------
def compute_recall(eval_data, retrieval_results):
    total = 0
    hit = 0

    # Normalize retrieval outputs into a dictionary
    retrieval_lookup = {
        normalize(item.get("query", "")): item
        for item in retrieval_results
    }

    for sample in eval_data:
        query = normalize(sample["query"])
        gold_chunks = set(sample["relevant_chunks"])

        total += 1

        # Check if this query has retrieval output
        if query not in retrieval_lookup:
            print(f"[WARN] No retrieved result for query: {sample['query']}")
            continue

        entry = retrieval_lookup[query]

        if "retrieved_chunks" not in entry:
            print(f"[WARN] Missing 'retrieved_chunks' in retrieval result for: {sample['query']}")
            continue

        # Extract retrieved chunk IDs
        retrieved_chunk_ids = {
            item["metadata"].get("chunk_id")
            for item in entry["retrieved_chunks"]
            if "metadata" in item
        }

        # Recall hit if ANY ground-truth chunk appears in retrieval list
        if gold_chunks.intersection(retrieved_chunk_ids):
            hit += 1

    return hit / total if total > 0 else 0

# ----------------------------
# RUN
# ----------------------------
eval_file = "rag_eval_dataset.json"
retrieval_file = "retrieval_output.json"

with open(eval_file, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

with open(retrieval_file, "r", encoding="utf-8") as f:
    retrieval_results = json.load(f)

recall = compute_recall(eval_data, retrieval_results)
print("Recall =", recall)


Recall = 0.7083333333333334


Generation

In [13]:
from langchain_community.llms import Ollama

# ---- Initialize Ollama ----
llm = Ollama(model="mistral:instruct")

# ---- Generation Results Container ----
generation_results = []

for item in retrieval_results:
    query = item["query"]

    # ---- Combine Retrieved Context ----
    context = "\n\n".join(
        [chunk["content"] for chunk in item["retrieved_chunks"]]
    )

    # ---- Prompt Template ----
    prompt = f"""
You are an academic assistant.
Answer the question using ONLY the provided context.
If the answer is not present, say:
"The information is not available in the provided documents."

Context:
{context}

Question:
{query}

Answer:
"""

    # ---- Generate Answer ----
    response = llm.invoke(prompt)

    generation_results.append({
        "id": item.get("id"),
        "query": query,
        "answer": response.strip(),
        "used_chunks": item["retrieved_chunks"]
    })

print(f"✅ Generated answers for {len(generation_results)} queries.")



with open("generation_output.json", "w", encoding="utf-8") as f:
    json.dump(generation_results, f, indent=2, ensure_ascii=False)



✅ Generated answers for 24 queries.


Judge

In [16]:
import json
import re
from langchain_community.llms import Ollama

# ---- Load Files ----
with open("rag_eval_dataset.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

with open("generation_output.json", "r", encoding="utf-8") as f:
    gen_data = json.load(f)

# ---- Map eval answers by query ----
eval_map = {item["query"]: item for item in eval_data}

judge_llm = Ollama(model="mistral:instruct")


def judge_prompt(question, reference, generated, context):
    return f"""
You are an impartial academic evaluator.

Evaluate the GENERATED ANSWER based on:
- the QUESTION
- the REFERENCE ANSWER
- the PROVIDED CONTEXT

Question:
{question}

Reference Answer:
{reference}

Generated Answer:
{generated}

Provided Context:
{context}

Evaluation Criteria:
1. Correctness – Does the generated answer match the reference answer?
2. Faithfulness – Is the answer supported by the context?
3. Completeness – Does it fully answer the question without missing key points?

Scoring Rules:
- Scores range from 1 (poor) to 5 (excellent).
- If the generated answer contradicts the reference, correctness ≤ 2.
- If the answer adds unsupported claims, faithfulness ≤ 2.

Return ONLY valid JSON:

{{
  "correctness": <int>,
  "faithfulness": <int>,
  "completeness": <int>,
  "verdict": "<short explanation>"
}}
"""

def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(match.group()) if match else None

judge_results = []

for item in gen_data:
    query = item["query"]
    generated = item["answer"]

    eval_item = eval_map.get(query)
    if not eval_item:
        continue

    reference = eval_item["answer"]

    context = "\n\n".join(
        chunk["content"] for chunk in item["used_chunks"]
    )

    prompt = judge_prompt(query, reference, generated, context)

    verdict_text = judge_llm.invoke(prompt)
    scores = extract_json(verdict_text)

    judge_results.append({
        "query": query,
        "reference_answer": reference,
        "generated_answer": generated,
        "judge_scores": scores
    })

print(f"✅ Judged {len(judge_results)} answers.")

with open("llm_judge_results.json", "w", encoding="utf-8") as f:
    json.dump(judge_results, f, indent=2, ensure_ascii=False)


✅ Judged 24 answers.


In [ ]:
# ===================== CELL 1: BLEU + BERTScore =====================

import json
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bertscore
import statistics

# ---- Download tokenizer ----
nltk.download("punkt")

# ---- Load Files ----
with open("mistral_rag_eval.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

with open("generation_output.json", "r", encoding="utf-8") as f:
    gen_data = json.load(f)

# ---- Map reference answers by query ----
eval_map = {item["query"]: item for item in eval_data}

smoothie = SmoothingFunction().method4

bleu_scores = {}
bertscore_scores = {}

generated_answers = []
reference_answers = []
queries = []

# ---- Compute BLEU + collect data for BERTScore ----
for item in gen_data:
    query = item["query"]
    generated = item["answer"]

    eval_item = eval_map.get(query)
    if not eval_item:
        continue

    reference = eval_item["answer"]

    # --- BLEU ---
    ref_tokens = [nltk.word_tokenize(reference.lower())]
    gen_tokens = nltk.word_tokenize(generated.lower())

    bleu = sentence_bleu(
        ref_tokens,
        gen_tokens,
        smoothing_function=smoothie
    )

    bleu_scores[query] = round(bleu, 4)

    # --- Store for BERTScore ---
    generated_answers.append(generated)
    reference_answers.append(reference)
    queries.append(query)

# ---- Compute BERTScore (Semantic Similarity) ----
P, R, F1 = bertscore(
    cands=generated_answers,
    refs=reference_answers,
    lang="en",
    model_type="microsoft/deberta-xlarge-mnli"  # strong, thesis-grade model
)

# ---- Map BERTScore back to queries ----
for i, q in enumerate(queries):
    bertscore_scores[q] = round(F1[i].item(), 4)

# ---- Save temporary results ----
with open("tmp_bleu.json", "w", encoding="utf-8") as f:
    json.dump(bleu_scores, f, indent=2)

with open("tmp_bertscore.json", "w", encoding="utf-8") as f:
    json.dump(bertscore_scores, f, indent=2)

print("✅ BLEU computed for", len(bleu_scores), "samples")
print("✅ BERTScore computed for", len(bertscore_scores), "samples")


In [ ]:
# ===================== CELL 2: LLM-As-Judge + FINAL OUTPUT =====================

import json
import re
import statistics
from langchain_community.llms import Ollama

# ---- Load Automatic Metrics ----
with open("tmp_bleu.json", "r", encoding="utf-8") as f:
    bleu_results = json.load(f)

with open("tmp_bertscore.json", "r", encoding="utf-8") as f:
    bertscore_results = json.load(f)

# ---- Reload Evaluation Data ----
with open("mistral_rag_eval.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

with open("generation_output.json", "r", encoding="utf-8") as f:
    gen_data = json.load(f)

eval_map = {item["query"]: item for item in eval_data}

# ---- Initialize Judge LLM ----
judge_llm = Ollama(model="mistral:instruct")


def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(match.group()) if match else None


def judge_prompt(question, reference, generated, context):
    return f"""
You are an impartial academic evaluator.

Evaluate the GENERATED ANSWER based on the QUESTION, REFERENCE ANSWER, and CONTEXT.

Question:
{question}

Reference Answer:
{reference}

Generated Answer:
{generated}

Provided Context:
{context}

Evaluation Criteria:
1. Correctness
2. Faithfulness
3. Completeness

Scoring Rules:
- Scores range from 1 (poor) to 5 (excellent).
- If the generated answer contradicts the reference, correctness ≤ 2.
- If unsupported claims appear, faithfulness ≤ 2.

Return ONLY valid JSON:

{{
  "correctness": <int>,
  "faithfulness": <int>,
  "completeness": <int>,
  "verdict": "<short explanation>"
}}
"""


results = []

# ---- LLM-as-Judge Evaluation Loop ----
for item in gen_data:
    query = item["query"]
    generated = item["answer"]

    eval_item = eval_map.get(query)
    if not eval_item:
        continue

    reference = eval_item["answer"]

    context = "\n\n".join(
        chunk["content"] for chunk in item["used_chunks"]
    )

    verdict_text = judge_llm.invoke(
        judge_prompt(query, reference, generated, context)
    )

    judge_scores = extract_json(verdict_text)

    results.append({
        "query": query,
        "reference_answer": reference,
        "generated_answer": generated,
        "bleu": bleu_results.get(query, 0.0),
        "bertscore": bertscore_results.get(query, 0.0),
        "judge_scores": judge_scores
    })


# ---- Save Final Output ----
with open("final_evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)


# ---- Summary Statistics ----
print(f"🏁 Final Evaluation Completed: {len(results)} samples")

print("Average BLEU:",
      round(statistics.mean(r["bleu"] for r in results), 4))

print("Average BERTScore:",
      round(statistics.mean(r["bertscore"] for r in results), 4))

print("Average Correctness:",
      round(statistics.mean(r["judge_scores"]["correctness"] for r in results), 2))

print("Average Faithfulness:",
      round(statistics.mean(r["judge_scores"]["faithfulness"] for r in results), 2))

print("Average Completeness:",
      round(statistics.mean(r["judge_scores"]["completeness"] for r in results), 2))


In [ ]:
# ===================== CELL: Visualization of RAG Evaluation =====================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import json

# Load the final evaluation results
with open("final_evaluation_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

# Check how many items are loaded
print(f"Total queries loaded: {len(results)}")

# Optional: view the first item
print(results[0])

# ---- Convert results to DataFrame ----
df = pd.DataFrame(results)


# Expand judge_scores dict into separate columns
judge_df = df['judge_scores'].apply(pd.Series)
df = pd.concat([df, judge_df], axis=1)

# ---- Chart 1: Average Scores Bar Chart ----
avg_scores = df[['bleu', 'bertscore', 'correctness', 'faithfulness', 'completeness']].mean()

plt.figure(figsize=(8,5))
sns.barplot(x=avg_scores.index, y=avg_scores.values, palette="viridis")
plt.title("Average Evaluation Scores")
plt.ylabel("Score")
plt.ylim(0,5)  # judge scores scale
plt.show()

# ---- Chart 2: Distribution Boxplot ----
plt.figure(figsize=(10,6))
sns.boxplot(data=df[['bleu', 'bertscore', 'correctness', 'faithfulness', 'completeness']], palette="coolwarm")
plt.title("Distribution of Evaluation Metrics")
plt.ylabel("Score")
plt.show()

# ---- Chart 3: Scatter Plot BLEU vs Faithfulness ----
plt.figure(figsize=(7,5))
sns.scatterplot(data=df, x='bleu', y='faithfulness')
plt.title("BLEU vs Faithfulness")
plt.xlabel("BLEU Score")
plt.ylabel("Faithfulness (LLM-as-Judge)")
plt.show()

# ---- Optional: Save Figures ----
plt.figure(figsize=(8,5))
sns.barplot(x=avg_scores.index, y=avg_scores.values, palette="viridis")
plt.title("Average Evaluation Scores")
plt.ylabel("Score")
plt.ylim(0,5)
plt.savefig("avg_scores_chart.png", dpi=300, bbox_inches='tight')
